# 03 Búsqueda vectorial con TF-IDF  
# Nombre: Wellington Barros
## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
import sys
import nltk
from nltk.corpus import gutenberg
import pandas as pd
import numpy as np

# Asegúrate de tener descargado el corpus
nltk.download('gutenberg')

# 1. Definir función de preprocesamiento (Usa la de tu Práctica 2)
def preprocesar(texto):
    # Ejemplo básico: minúsculas y tokenización alfanumérica
    import re
    return re.findall(r'\b[a-z]+\b', texto.lower())

# 2. Cargar documentos
documentos = []
for fileid in gutenberg.fileids():
    texto_original = gutenberg.raw(fileid)
    tokens = preprocesar(texto_original)
    terminos_unicos = len(set(tokens))
    documentos.append({
        'id': fileid,
        'titulo': fileid.replace('.txt', ''), # Aproximación al título
        'texto_original': texto_original,
        'tokens': tokens,
        'num_terminos_unicos': terminos_unicos,
        'longitud': len(tokens)
    })

df_corpus = pd.DataFrame(documentos)

# 3. Mostrar métricas solicitadas
num_documentos = len(df_corpus)
vocabulario_total = set([token for doc in df_corpus['tokens'] for token in doc])
tamano_vocabulario = len(vocabulario_total)

longitud_min = df_corpus['longitud'].min()
longitud_media = df_corpus['longitud'].mean()
longitud_max = df_corpus['longitud'].max()

print(f"Número de documentos: {num_documentos}")
print(f"Tamaño del vocabulario: {tamano_vocabulario}")
print(f"Longitud mínima: {longitud_min} | Media: {longitud_media:.2f} | Máxima: {longitud_max}")

# Mostrar tabla
df_corpus[['id', 'titulo', 'num_terminos_unicos']].head()

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


Número de documentos: 18
Tamaño del vocabulario: 41481
Longitud mínima: 6934 | Media: 118632.94 | Máxima: 791842


,id,titulo,num_terminos_unicos
0,austen-emma.txt,austen-emma,7079
1,austen-persuasion.txt,austen-persuasion,5739
2,austen-sense.txt,austen-sense,6283
3,bible-kjv.txt,bible-kjv,12568
4,blake-poems.txt,blake-poems,1510


Al analizar el corpus de Gutenberg, se observo que está compuesto por una cantidad fija de documentos clásicos que varían drásticamente en su longitud, desde textos cortos hasta obras sumamente extensas. El tamaño del vocabulario global es representativo de la riqueza léxica del inglés literario. Esta variabilidad en la longitud de los documentos justifica la necesidad futura de normalizar los vectores (como en la similitud coseno), ya que los documentos más largos naturalmente tendrán frecuencias de términos más altas por su mera extensión, no necesariamente por ser más relevantes.

### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [2]:
import math

def calcular_tf(tokens_doc):
    """Devuelve un diccionario con la frecuencia (TF) de cada término en el documento."""
    tf_dict = {}
    for token in tokens_doc:
        tf_dict[token] = tf_dict.get(token, 0) + 1
    return tf_dict

def calcular_df(lista_tokens_por_doc):
    """Calcula en cuántos documentos aparece cada término (DF)."""
    df_dict = {}
    for tokens in lista_tokens_por_doc:
        terminos_unicos = set(tokens)
        for token in terminos_unicos:
            df_dict[token] = df_dict.get(token, 0) + 1
    return df_dict

def calcular_idf(df_dict, N):
    """Calcula el IDF aplicando logaritmo natural + 1."""
    idf_dict = {}
    for term, df in df_dict.items():
        idf_dict[term] = math.log(N / df) + 1.0
    return idf_dict

def calcular_pesos_tfidf(tf_dict, idf_dict):
    """Multiplica TF por IDF."""
    tfidf_dict = {}
    for term, tf in tf_dict.items():
        if term in idf_dict:
            tfidf_dict[term] = tf * idf_dict[term]
    return tfidf_dict

Las funciones implementadas reflejan matemáticamente la penalización de términos comunes. El $TF$ captura la importancia local de una palabra en un texto, mientras que el $IDF$ actúa como un filtro global. Al usar la fórmula con logaritmo natural y sumar 1 (estándar de scikit-learn sin suavizado), se evita que los términos que aparecen en todos los documentos tengan un peso de 0, asegurando que su presencia aún aporte un mínimo de información al modelo.

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [3]:
# 1. Vocabulario con orden definido
vocab_ordenado = sorted(list(vocabulario_total))
term_to_index = {term: idx for idx, term in enumerate(vocab_ordenado)}

# 2. Contar DF y calcular IDF
df_diccionario = calcular_df(df_corpus['tokens'])
idf_diccionario = calcular_idf(df_diccionario, num_documentos)

# 3. TF y TF-IDF por documento
matriz_tfidf_manual = np.zeros((num_documentos, tamano_vocabulario))

for i, doc_tokens in enumerate(df_corpus['tokens']):
    tf_dic = calcular_tf(doc_tokens)
    tfidf_dic = calcular_pesos_tfidf(tf_dic, idf_diccionario)
    
    # Llenar la matriz
    for term, peso in tfidf_dic.items():
        idx = term_to_index[term]
        matriz_tfidf_manual[i, idx] = peso

print(f"Dimensiones de la matriz TF-IDF manual: {matriz_tfidf_manual.shape}")

Dimensiones de la matriz TF-IDF manual: (18, 41481)


Se transformó exitosamente una colección de textos en una matriz matemática densa (documento-término). En esta estructura, cada fila representa un documento en un espacio vectorial de miles de dimensiones (el vocabulario). Las celdas con valor cero indican la ausencia del término, evidenciando que esta matriz es altamente dispersa (sparse), una característica típica en la Recuperación de Información.

### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Unir tokens con espacios para pasarlos al vectorizador (simulando texto preprocesado)
corpus_unido = [" ".join(doc) for doc in df_corpus['tokens']]

# Configuración estricta pedida en la práctica
vectorizer = TfidfVectorizer(
    vocabulary=vocab_ordenado, # Forzamos el mismo vocabulario y orden
    token_pattern=r"(?u)\b\w+\b", # Para no descartar palabras de 1 sola letra si las hay
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None
)

matriz_tfidf_sklearn = vectorizer.fit_transform(corpus_unido).toarray()

# Calcular diferencia máxima absoluta
diferencia = np.abs(matriz_tfidf_manual - matriz_tfidf_sklearn)
max_dif = np.max(diferencia)

print(f"Máxima diferencia absoluta entre manual y sklearn: {max_dif}")
if max_dif < 1e-9:
    print("¡Las matrices son prácticamente idénticas!")

Máxima diferencia absoluta entre manual y sklearn: 0.0
¡Las matrices son prácticamente idénticas!


La obtención de una diferencia absoluta de cero entre la matriz calculada de forma manual y la generada por TfidfVectorizer valida rigurosamente la precisión matemática de nuestra implementación. Este resultado confirma que las funciones diseñadas para el conteo de frecuencias locales (
TF), el cálculo global de documentos por término (
DF), la aplicación del logaritmo natural con ajuste de unidad en el 
IDF y la ponderación final operan con exacta idénticidad respecto a la librería estándar de la industria, cumpliendo cabalmente con las restricciones de configuración estrictas solicitadas.

### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Definir 4 consultas (ejemplo)
consultas = [
    "whale sea captain",          # Q1: Moby Dick (aventura marina)
    "alice rabbit wonderland",    # Q2: Alicia en el País de las Maravillas
    "pride prejudice marriage",   # Q3: Orgullo y Prejuicio (temas sociales)
    "science astronomy physics"   # Q4: Ciencia general
]

def procesar_consulta(query, vocab_idx, idf_dict):
    tokens_q = preprocesar(query)
    tf_q = calcular_tf(tokens_q)
    tfidf_q = calcular_pesos_tfidf(tf_q, idf_dict)
    
    vec = np.zeros(len(vocab_idx))
    for term, peso in tfidf_q.items():
        if term in vocab_idx:
            vec[vocab_idx[term]] = peso
    return vec, tokens_q

# Tomar la primera consulta y el primer documento para mostrar el cálculo manual
q_vec, tokens_q1 = procesar_consulta(consultas[0], term_to_index, idf_diccionario)
d_vec = matriz_tfidf_manual[0] # Documento 0

# Similitud Coseno Manual
producto_punto = np.dot(q_vec, d_vec)
norma_q = np.linalg.norm(q_vec)
norma_d = np.linalg.norm(d_vec)
similitud_manual = producto_punto / (norma_q * norma_d) if (norma_q * norma_d) > 0 else 0

# Similitud Coseno Sklearn
similitud_sklearn = cosine_similarity(q_vec.reshape(1, -1), d_vec.reshape(1, -1))[0][0]

# Tabla de demostración para Q1 y D0
compartidos = [t for t in tokens_q1 if d_vec[term_to_index.get(t, -1)] > 0]
print(f"Consulta: '{consultas[0]}' vs Documento ID: {df_corpus.iloc[0]['id']}")
print(f"Términos compartidos: {compartidos}")
print(f"Producto punto: {producto_punto:.4f}")
print(f"Norma Q: {norma_q:.4f} | Norma D: {norma_d:.4f}")
print(f"Similitud Manual: {similitud_manual:.6f}")
print(f"Similitud Sklearn: {similitud_sklearn:.6f}")

Consulta: 'whale sea captain' vs Documento ID: austen-emma.txt
Términos compartidos: ['sea', 'captain']
Producto punto: 26.0924
Norma Q: 2.9190 | Norma D: 15320.0072
Similitud Manual: 0.000583
Similitud Sklearn: 0.000583



La similitud coseno mide el ángulo entre el vector de la consulta y el vector del documento, independientemente de sus magnitudes. Esto es vital: un documento no se vuelve "menos relevante" solo por ser más largo (lo que inflaría sus frecuencias TF y su norma vectorial). Como vimos en la demostración manual y la de la librería, el producto punto captura la coincidencia de pesos, mientras que la división por las normas escala el resultado estrictamente entre 0 y 1

### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [6]:
# Transformar todas las consultas con sklearn (usando el vectorizador ajustado)
consultas_unidas = [" ".join(preprocesar(q)) for q in consultas]
matrices_consultas = vectorizer.transform(consultas_unidas)

# Calcular similitudes de todas las consultas contra todos los documentos
similitudes = cosine_similarity(matrices_consultas, matriz_tfidf_manual)

for i, query in enumerate(consultas):
    print(f"\n--- Ranking para la consulta: '{query}' ---")
    
    # Obtener índices ordenados de mayor a menor similitud
    indices_ordenados = np.argsort(similitudes[i])[::-1]
    
    print(f"{'Pos':<4} | {'Identificador':<20} | {'Puntuación':<10} | Fragmento")
    print("-" * 80)
    
    # Mostrar Top-5
    for rank in range(5):
        doc_idx = indices_ordenados[rank]
        score = similitudes[i][doc_idx]
        doc_id = df_corpus.iloc[doc_idx]['id']
        
        # Extraer un pequeño fragmento (primeros 60 caracteres)
        fragmento = df_corpus.iloc[doc_idx]['texto_original'][:60].replace('\n', ' ')
        
        print(f"{rank+1:<4} | {doc_id:<20} | {score:<10.6f} | {fragmento}...")


--- Ranking para la consulta: 'whale sea captain' ---
Pos  | Identificador        | Puntuación | Fragmento
--------------------------------------------------------------------------------
 (Supp...Y.ille-moby_dick.txt | 0.109914   | [Moby Dick by Herman Melville 1851]
2    | austen-persuasion.txt | 0.038019   | [Persuasion by Jane Austen 1818]   Chapter 1   Sir Walter El...
3    | whitman-leaves.txt   | 0.007754   | [Leaves of Grass by Walt Whitman 1855]   Come, said my soul,...
...  | bryant-stories.txt   | 0.006693   | [Stories to Tell to Children by Sara Cone Bryant 1918] 
5    | chesterton-brown.txt | 0.006133   | [The Wisdom of Father Brown by G. K. Chesterton 1914]   I. T...

--- Ranking para la consulta: 'alice rabbit wonderland' ---
Pos  | Identificador        | Puntuación | Fragmento
--------------------------------------------------------------------------------
1    | carroll-alice.txt    | 0.236411   | [Alice's Adventures in Wonderland by Lewis Carroll 1865]  CH...
...  | 

El modelo vectorial con ponderación TF-IDF genera rankings sumamente lógicos. Al evaluar el Top-5 de cada consulta, se observa que los documentos que tratan la temática principal de la búsqueda (como Moby Dick para la consulta sobre ballenas) obtienen puntajes mucho más altos. Esto demuestra la superioridad del modelo vectorial frente al booleano tradicional (Ejercicio 2): no solo sabemos qué documentos contienen los términos, sino que podemos ordenarlos matemáticamente según qué tan centrales son esos términos para el texto, gracias al equilibrio entre la frecuencia local (TF) y la rareza global de la palabra (IDF).